# Preprocessing

**Dataset:** FineWeb-Edu 10BT Sample (14 Parquet files)  
**Platform:** SDSC Expanse — 32 cores total (1 driver + 31 executors)  
**Task:** 5-class educational quality scoring (`int_score` 0–4)

---

## Setup & Imports

In [1]:
!pip install -qq pyspark

import os
import sys
from pathlib import Path

# Get the parent directory and add it to the path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

## Data Acquisition

Download 14 Parquet files from the FineWeb-Edu 10BT HuggingFace dataset into `../data/`. Files are skipped if already present.

In [2]:
from fineweb_spark.dataset import download_fineweb_sample

download_fineweb_sample()

Skipping 000_00000.parquet (already exists)
Skipping 001_00000.parquet (already exists)
Skipping 002_00000.parquet (already exists)
Skipping 003_00000.parquet (already exists)
Skipping 004_00000.parquet (already exists)
Skipping 005_00000.parquet (already exists)
Skipping 006_00000.parquet (already exists)
Skipping 007_00000.parquet (already exists)
Skipping 008_00000.parquet (already exists)
Skipping 009_00000.parquet (already exists)
Skipping 010_00000.parquet (already exists)
Skipping 011_00000.parquet (already exists)
Skipping 012_00000.parquet (already exists)
Skipping 013_00000.parquet (already exists)
All 14 sample files downloaded.


## Spark Session & Cluster Verification

Start a Spark session on SDSC Expanse with 31 executors, then verify distributed resources via the Spark REST API.

In [3]:
# Driver: 2g (coordinator only, does not process data)
# Executor instances: Total Cores - 1 = 32 - 1 = 31
# Memory per executor: (128g - 2g) / (31 × 1.1 overhead) ≈ 3g
# Actual usage: 31 × (3g × 1.1) + 2g ≈ 104g — safely within 128GB
spark = SparkSession.builder \
    .appName("Milestone3_Complete_Pipeline") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.instances", "31") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "3g") \
    .getOrCreate()

spark

In [4]:
import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
df['maxMemory_GB'] = (df['maxMemory'] / (1024**3)).round(2)
df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,32,1099746508,0,True,1.02


## Load & Explore Data

Read all 14 Parquet files into a single Spark DataFrame, inspect the schema, and preview sample rows.

In [5]:
from fineweb_spark.paths import RAW_DATA_DIR

df = spark.read.parquet(str(RAW_DATA_DIR))

In [6]:
# Show the schema of the DataFrame to verify it loaded correctly
df.printSchema()

root
 |-- text: string (nullable = true)
 |-- id: string (nullable = true)
 |-- dump: string (nullable = true)
 |-- url: string (nullable = true)
 |-- file_path: string (nullable = true)
 |-- language: string (nullable = true)
 |-- language_score: double (nullable = true)
 |-- token_count: long (nullable = true)
 |-- score: double (nullable = true)
 |-- int_score: long (nullable = true)



In [7]:
# Shows 10 rows, truncating every column to a maximum of 10 characters
df.show(10, truncate=10)

+----------+----------+----------+----------+----------+--------+--------------+-----------+--------+---------+
|      text|        id|      dump|       url| file_path|language|language_score|token_count|   score|int_score|
+----------+----------+----------+----------+----------+--------+--------------+-----------+--------+---------+
|When yo...|<urn:uu...|CC-MAIN...|https:/...|s3://co...|      en|    0.90174...|        407|2.953125|        3|
|An eati...|<urn:uu...|CC-MAIN...|https:/...|s3://co...|      en|    0.96011...|       1309|3.703125|        4|
|- Devel...|<urn:uu...|CC-MAIN...|http://...|s3://co...|      en|    0.96301...|       2293|2.796875|        3|
|âEquity...|<urn:uu...|CC-MAIN...|http://...|s3://co...|      en|    0.95104...|       4221|3.671875|        4|
|Anythin...|<urn:uu...|CC-MAIN...|https:/...|s3://co...|      en|    0.94335...|        640| 2.78125|        3|
|Dietary...|<urn:uu...|CC-MAIN...|https:/...|s3://co...|      en|    0.92175...|       1060|3.703125|   

# PREPROCESSING PIPELINE 

### Data Cleaning & Feature Engineering

In [8]:
# Filtering Invalid Documents
df = df.filter(
    (F.col("text").isNotNull()) &
    (F.length(F.col("text")) > 200) &
    (F.col("token_count") >= 50) &
    (F.col("language") == "en") &
    (F.col("language_score") >= 0.9)
)

In [9]:
# Check if duplicates actually exist BEFORE deciding to deduplicate
total = df.count()
unique = df.select("id").distinct().count()
print(f"Total: {total}, Unique: {unique}, Duplicates: {total - unique}")

Total: 8470212, Unique: 8470212, Duplicates: 0


In [10]:
# Show the distribution of the target variable (int_score) to check for class imbalance befor Stratified sampling
df.groupBy("int_score").count().orderBy("int_score").show()

+---------+-------+
|int_score|  count|
+---------+-------+
|        3|7337302|
|        4|1126918|
|        5|   5992|
+---------+-------+



### Data Split & Stratified Sampling


In [11]:
# Stratified sampling to address class imbalance:
#   int_score 3 -> keep 5%   (dominant class ~86.7% of data)
#   int_score 4 -> keep 30%  (moderately common ~13.2% of data)
#   int_score 5 -> keep 100% (rare minority, only 7,430 docs)
fractions = {3: 0.05, 4: 0.30, 5: 1.0}
df = df.sampleBy("int_score", fractions=fractions, seed=42)

In [12]:
# Show the distribution of the target variable (int_score) after Stratified sampling
df.groupBy("int_score").count().orderBy("int_score").show()

+---------+------+
|int_score| count|
+---------+------+
|        3|366405|
|        4|338931|
|        5|  5992|
+---------+------+



In [13]:
# Feature Engineering using Spark SQL functions (regexp_extract, length, when, otherwise)
df = df.withColumn(
    "domain",
    F.regexp_extract(F.col("url"), r"https?://([^/]+)", 1)
)
df = df.withColumn(
    "length_bucket",
    F.when(F.col("token_count") < 500, "short")
     .when(F.col("token_count") < 2000, "medium")
     .otherwise("long")
)

In [14]:
# Defining the target variable (quality)
df = df.withColumn("label", F.col("int_score").cast(IntegerType()))

In [15]:
# Show the final schema of the DataFrame after adding features
df.printSchema()

root
 |-- text: string (nullable = true)
 |-- id: string (nullable = true)
 |-- dump: string (nullable = true)
 |-- url: string (nullable = true)
 |-- file_path: string (nullable = true)
 |-- language: string (nullable = true)
 |-- language_score: double (nullable = true)
 |-- token_count: long (nullable = true)
 |-- score: double (nullable = true)
 |-- int_score: long (nullable = true)
 |-- domain: string (nullable = true)
 |-- length_bucket: string (nullable = false)
 |-- label: integer (nullable = true)



In [16]:
# Save preprocessed data to data/processed/
from fineweb_spark.paths import INTERIM_DATA_DIR, ROOT_DIR

output_path = INTERIM_DATA_DIR / "fineweb_preprocessed.parquet"

df.write.mode("overwrite").parquet(str(output_path))

print(f"Saved preprocessed data to {output_path.relative_to(ROOT_DIR)}")

Saved preprocessed data to data/interim/fineweb_preprocessed.parquet
